In [ ]:
%%capture
!pip install transformers sentencepiece bert_score sacrebleu nltk accelerate

# Overview

This notebook will walk you through the steps to carrying out a machine translation evaluation. It will cover the process of uploading a dataset, selecting a model, and then carrying out the evaluation over a set of chosen metrics.

Most of this notebook is automated, however, there will be certain actions required of the user. These will be highlighted as necessary.

**Action:**
Ensure this notebook runtime is set to a GPU (T4, A100, V100)

Your Notebook session *may* produce the following error:

```OutOfMemoryError: CUDA out of memory. Tried to allocate ...```

This generally happens when re-running one of the translation models. When this happens just restart the runtime. Try to only run the translation once and then go straight onto the evaluation. When evaluating a second model, restart the runtime again, this time using the second model for translation and not running the first.


# Step 0 - Setup
Run the code below to mount your Google Drive and most of the necessary packages to carry out the evaluation

**Action:**
No code changes required. When prompted, connect your Google account

In [ ]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install bert_score

In [ ]:
%%capture
!cd /content/
!rm -rf ./CASM_utils/
!pip install -e CASM_utils/
!cd /content/CASM_utils


import CASM_utils
import importlib
from CASM_utils import utils
importlib.reload(utils)

In [ ]:
## Mount GDrive

## Imports
import nltk
import time
import torch
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from transformers import pipeline
from transformers import AutoTokenizer
from bert_score import score as bert_score
from torch.utils.data import DataLoader, Dataset
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nltk.download('wordnet')
nltk.download('wordnet_ic')
nltk.download('punkt')

# Step 1 - Upload a translation dataset
The first task will be to find a dataset, or several, that are suitable for the machine translation evaluation task.

**Action:**

In order to make the dataset compatible for the evaluation, ensure that there are distinct columns for the source language ('source') and translation target ('target'):
- 'source': The column representing the text in its original language
- 'target': The column representing the gold standard translation (in English)

**If your dataset is NOT in this format, please format it in a spreadsheet and then export it to a csv before uploading, alternatively you can create a new code cell to make the necessary changes.**

To ensure evaluations are done quickly, please filter the dataset down to no more than 1000 rows

Dataset resources:

(*Be cautious when using datasets from the Helsinki repo, as the training data would have been used to train the opus models which are commonly used for translation*)

- https://datasetsearch.research.google.com/
- https://github.com/Helsinki-NLP/OPUS-MT-train/tree/master/models
- https://github.com/Helsinki-NLP/Tatoeba-Challenge/tree/master/models


In [ ]:
# direct the file handler to the data folder
# FOLDER = '/content/drive/MyDrive/PROJECTs/00-FAST/01-Machine_Translation'  # ← replace with your local path
fh = utils.FileHandler(FOLDER)

In [ ]:
# Use this code cell to upload datasets, carry out any pre-processing
#   and format them as DataFrames with the columns ['source', 'target']

### Dataset 1 pre-processing
# https://www.statmt.org/europarl/
europarl_corpus = fh.csv_to_df(
    'evaluation_data/europarl_corpus.csv'
)
dataset_1 = europarl_corpus.sample(1000, random_state=1)


### Dataset 2 pre-processing
# https://www.statmt.org/europarl/
opus_corpus = fh.csv_to_df(
    'evaluation_data/opus_corpus.csv'
)
dataset_2 = opus_corpus.sample(1000, random_state=1)

In [ ]:
dataset = dataset_2 # Change this line of code so that the evaluation dataset is being assigned
dataset_name = 'opus'
model_name = 'Helsinki'

# The following lines will produce errors if the dataset has not been formatted correctly
assert 'source' in dataset.columns
assert 'target' in dataset.columns
assert len(dataset['source']) == len(dataset['target'])

dataset.head()

# Step 2 - Select a machine translation model

There may be several candidate models for machine translation so we will implement them below. In this notebook will be a template for the following:
- **Facebook MBART-Large-50-many-to-many-mmt**
- **Helsinki-NLP Opus series**

These models do not cover *all* languages so additional scoping may be needed. Like with the above datasets, we will be evaluating these models one at a time so it is important that **not all** code cells are run, as you will be overwriting the dataset.

## Helsinki-NLP/opus-mt-XX-XX
The Helsinki OPUS series covers a wide range of language pairs. In order to adapt this code you will need to find a HuggingFace🤗 model card for your specific pair. E.g:

- https://huggingface.co/Helsinki-NLP/opus-mt-zh-en (Chinese -> English)
- https://huggingface.co/Helsinki-NLP/opus-mt-fr-en (French -> English)

You have two ways of finding this. First, and the most simplest, is to Google "Helsinki opus [language code]-en" and it will usually be the top result. If you do not know the language code you can use the HuggingFace🤗 website (use one of the above links). Select the model button at the top of the page, in tasks tab select 'Translation', and then switch to the language tab and select both the source language, and the target (English). This will then identify all the relevant results, of which the Helsinki models can be found.

**Action:**
First follow the above steps to identify a Helsinki-NLP model for your language pair. Then, adapt the below code so that the link to the model is updated to match the model you have found.

In [ ]:
# !! ACTION: Update the below variable to match the model for your language!!
# !! You can do this by copying the end part of the URL,
# !! e.g: https://huggingface.co/Helsinki-NLP/opus-mt-es-en -> Helsinki-NLP/opus-mt-es-en
current_model = 'Helsinki-NLP/opus-mt-es-en'

tokenizer = AutoTokenizer.from_pretrained(current_model)
model = AutoModelForSeq2SeqLM.from_pretrained(current_model)

class TranslationDataset(Dataset):
  def __init__(self, data, tokenizer, max_length):
    self.data = data
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    article = self.data[idx]
    encoded = self.tokenizer(
        article, return_tensors="pt", padding="max_length",
        truncation=True, max_length=self.max_length
    )
    return {
      'input_ids': encoded['input_ids'].squeeze(),
      'attention_mask': encoded['attention_mask'].squeeze()
    }

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

translation_dataset = TranslationDataset(dataset['source'].tolist(), tokenizer, max_length=512)
dataloader = DataLoader(translation_dataset, batch_size=16, shuffle=False)

translations = []

log_batch = 16
log_gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader,nounits"]).decode("utf-8").strip()
last_time = dt.datetime.today().timestamp()
start_time = dt.datetime.today().timestamp()
diffs = []
for batch in tqdm(dataloader):
    data = {k: v.to('cuda') for k, v in batch.items()}

    generated_tokens = model.generate(
        **data
    )
    batch_translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    translations.extend(batch_translation)

    # Time logging
    new_time = dt.datetime.today().timestamp()
    diffs.append(new_time - last_time)
    last_time = new_time

end_time = dt.datetime.today().timestamp()
log_iters = round(np.mean(diffs), 2)
log_duration = round(end_time - start_time, 2)
log_memory = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"]).decode("utf-8").strip()

dataset['machine_translation'] = translations

report = f"""
MODEL: {current_model}.
Translation complete using GPU[{log_gpu}] using batch size[{log_batch}].
For a dataset of [{len(dataset)}] samples the translation took [{log_duration}] seconds with an average speed of [{log_iters}] iterations per second.
Memory usage: {log_memory} MiB.
"""
print(report)

# Free up memory
torch.cuda.empty_cache()

In [ ]:
# Writing the string to the file
with open(fh.cr_fn(f'evaluation_output/{model_name}/{dataset_name}/computational_results.txt'), 'w') as file:
    file.write(report)

The above code will produce the translations using the configured model.

**Action:** Copy the above string "Translation complete using..." into the evaluation document in the necessary section. This is to ensure we can have a rough estimate of throughput and how it will scale with larger datasets.

## Custom

If neither of the above models have capacity to translate your specific language, you have two options. First, if there is a model for another language of the same family, you can use that instead (for example: You can use the Indonesian model for Malaysian translation). Alternatively, you will need to source a new model from either HuggingFace🤗 or elsewhere.

When setting up a model you will need to, if possible, find a way to record the model speed, along with the GPU being used, and the batch size. This is so that we can estimate how long it will take to use the model on datasets of varying size.

Ensure that the variable ```current_model``` is updated to store a string representing the model name, and that the translations are stored in the dataset with the correct heading. E.g: ```dataset['machine_translation'] = translations```

# Step 3 - Evaluation metrics
The code in the following sections should be able to run with no alterations. Each subsection will produce an evaluation metrics which will be aggregated towards the end to make it easier to fill out the translation evaluation document.

## METEOR score

In [ ]:
meteor = nltk.translate.meteor_score.meteor_score
scores = []
for ref, hyp in tqdm(zip(dataset['target'].values, dataset['machine_translation'].values)):
    ref_tokens = ref.split()
    hyp_tokens = hyp.split()
    score = meteor([ref_tokens], hyp_tokens)
    scores.append(score)

# Average the METEOR scores
average_score = np.mean(scores)

# Print the average METEOR score
dataset['meteor_score'] = scores
print("Average METEOR score:", average_score)

## BERTScore


In [ ]:

references = dataset['target'].tolist()
cands = dataset['machine_translation'].tolist()

tokenizer = AutoTokenizer.from_pretrained('roberta-large')

references_tokenized = [" ".join(tokenizer.tokenize(ref)) for ref in references]
cands_tokenized = [" ".join(tokenizer.tokenize(cand)) for cand in cands]

P, R, F1 = bert_score(cands_tokenized, references_tokenized, model_type = 'roberta-large', lang="en", verbose=True)

# Average the BERTScore F1 scores
average_F1 = F1.mean().item()

# Print the average BERTScore F1
dataset['bert_score'] = F1
print("Average BERTScore F1:", average_F1)

## Mulitilingual BERTScore


In [ ]:
references = dataset['target'].tolist()
cands = dataset['machine_translation'].tolist()

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-large-mnli')

references_tokenized = [" ".join(tokenizer.tokenize(ref)) for ref in references]
cands_tokenized = [" ".join(tokenizer.tokenize(cand)) for cand in cands]

P, R, F1 = bert_score(cands_tokenized, references_tokenized, model_type = 'microsoft/deberta-large-mnli', verbose=True)

# Average the BERTScore F1 scores
average_F1 = F1.mean().item()

# Print the average BERTScore F1
dataset['bilingual_score'] = F1
print("Average BERTScore F1:", average_F1)

## Summary

**Action:**
Copy over the mean scores for each metric over to the translation evaluation document.

In [ ]:
results = pd.DataFrame()
results['meteor_score'] = dataset['meteor_score'].describe()
results['bert_score'] = dataset['bert_score'].describe()
results['bilingual_score'] = dataset['bilingual_score'].describe()
print(current_model)
display(results)
results.to_csv(
    fh.cr_fn(f'evaluation_output/{model_name}/{dataset_name}/{dataset_name}_results.csv'),
)

**Error analysis**
To supplement the evaluation, a human annotator should go through the translation results to make a human judgment on quality. Some of the details to look out for are:
- How well are named entities preserved?
- How well are semantics preserved?
- What types of text is the model struggling on? (Does it struggle with links, emojis, code-switching?)

The below code cell will give a preview of the lowest scoring translations for each metrics, but will also output a spreadsheet for an analyst to go through.

**Action:**
Ensure the filepath is updated to prevent files from being overwritten

In [ ]:
# !! Change this filepath !!
dataset.to_csv(
    fh.cr_fn(f'evaluation_output/{model_name}/{dataset_name}/{dataset_name}_mt_output.csv'),
    index=False,
)

In [ ]:
dataset.sample(n=100, random_state=1).to_csv(
    fh.cr_fn(f'evaluation_output/{model_name}/{dataset_name}/{dataset_name}_mt_samples.csv'),
    index=False
)

In [ ]:
def check_tokenisation(tokenizer, sentences):
    # Tokenize sentences and analyze truncation
    tokenized_data = []
    for sentence in tqdm(sentences):
        tokens = tokenizer.tokenize(sentence)
        tokenized_length = len(tokens)
        truncated = tokenized_length > tokenizer.model_max_length
        truncation_amount = max(0, tokenized_length - tokenizer.model_max_length)

        tokenized_data.append({
            "sentence": sentence,
            "num_tokens": tokenized_length,
            "truncated": truncated,
            "truncation_amount": truncation_amount
        })

    # Convert to DataFrame for better visualization
    tokenized_df = pd.DataFrame(tokenized_data)
    return tokenized_df

In [ ]:
source = dataset['source']
target = dataset['target']

In [ ]:
check_tokenisation(tokenizer, target)